In [12]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
import numpy as np
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [13]:
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """
    Découpe X et y en trois jeux :
    train, validation et test.

    Retourne :
    X_train, X_val, X_test, y_train, y_val, y_test
    """

    if not (0 <= test_size < 1):
        raise ValueError("test_size doit être compris entre 0 et 1.")

    if not (0 <= val_size < 1):
        raise ValueError("val_size doit être compris entre 0 et 1.")

    if test_size + val_size >= 1:
        raise ValueError("test_size + val_size doit être strictement inférieur à 1.")

    # 1. Isolation du jeu de test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    # Cas particulier : pas de validation
    if val_size == 0:
        return X_temp, None, X_test, y_temp, None, y_test

    # 2. Calcul de la proportion de validation sur ce qu'il reste
    val_ratio_on_remaining = val_size / (1 - test_size)

    # 3. Séparation train / validation
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=val_ratio_on_remaining,
        stratify=y_temp,
        random_state=random_state
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [14]:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y,
    test_size=0.2,
    val_size=0.16 
)

print(f"Train : {len(X_train)}")
print(f"Validation : {len(X_val)}")
print(f"Test : {len(X_test)}")

def proportions(y):
    classes, counts = np.unique(y, return_counts=True)
    return counts / len(y)

print("Global      :", proportions(y))
print("Train       :", proportions(y_train))
print("Validation  :", proportions(y_val))
print("Test        :", proportions(y_test))

Train : 364
Validation : 91
Test : 114
Global      : [0.37258348 0.62741652]
Train       : [0.37362637 0.62637363]
Validation  : [0.37362637 0.62637363]
Test        : [0.36842105 0.63157895]


In [15]:
print( len(X_train) + len(X_val) + len(X_test) == len(X))

True


In [16]:
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, val_size=0
)

print(X_val is None)
print(y_val is None)

True
True


In [17]:
y_imb = np.array([0] * 950 + [1] * 50)
X_imb = np.random.randn(1000, 5)

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X_imb, y_imb
)

for nom, y_split in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    prop = np.mean(y_split)
    print(f"{nom}: {prop:.3f}")

Train: 0.050
Validation: 0.050
Test: 0.050


In [18]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42,replace=True):
    """
    Évalue la stabilité d'un modèle par bootstrap.

    Pour chaque itération :
    - tire un échantillon bootstrap (avec remise)
    - entraîne le modèle sur cet échantillon
    - évalue sur les points non tirés (out-of-bag)

    Retourne la liste des scores.
    """

    rng = np.random.default_rng(random_state)
    n_samples = len(X)

    scores = []

    for _ in range(n_iterations):

        # Tirage bootstrap avec remise
        bootstrap_idx = rng.choice(
            n_samples,
            size=n_samples,
            replace=replace
        )

        # Indices OOB (jamais tirés)
        oob_mask = np.ones(n_samples, dtype=bool)
        oob_mask[bootstrap_idx] = False
        oob_idx = np.where(oob_mask)[0]

        # Cas rare : aucun point OOB
        if len(oob_idx) == 0:
            continue

        X_train = X[bootstrap_idx]
        y_train = y[bootstrap_idx]

        X_test = X[oob_idx]
        y_test = y[oob_idx]

        # Clone pour repartir d'un modèle vierge
        model = clone(modele)

        model.fit(X_train, y_train)

        score = model.score(X_test, y_test)
        scores.append(score)

    if len(scores) == 0:
        print("Aucun score calculable (tous les échantillons OOB étaient vides).")
        return []

    moyenne = np.mean(scores)

    if len(scores) > 1:
        ecart_type = np.std(scores)
        print(
            f"Score moyen sur {len(scores)} bootstraps : "
            f"{moyenne:.3f} (± {ecart_type:.3f})"
        )
    else:
        print(
            f"Score sur l'unique bootstrap : {moyenne:.3f}"
        )

    return scores

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

tree = DecisionTreeClassifier(random_state=42)
logreg = LogisticRegression(max_iter=5000)

print("\n=== Decision Tree ===")
scores_tree = bootstrap_scores(tree, X, y, n_iterations=30, random_state=42)

print("\n=== Logistic Regression ===")
scores_lr = bootstrap_scores(logreg, X, y, n_iterations=30, random_state=42)




=== Decision Tree ===


Score moyen sur 30 bootstraps : 0.923 (± 0.018)

=== Logistic Regression ===
Score moyen sur 30 bootstraps : 0.952 (± 0.015)


In [20]:
print("\n=== Decision Tree replace = false ===")
scores_tree = bootstrap_scores(tree, X, y, n_iterations=30, random_state=42,replace=False)


=== Decision Tree replace = false ===
Aucun score calculable (tous les échantillons OOB étaient vides).


In [21]:
print("\n=== Decision Tree n_iterations=1 ===")
scores_tree = bootstrap_scores(tree, X, y, n_iterations=1, random_state=42)



=== Decision Tree n_iterations=1 ===
Score sur l'unique bootstrap : 0.882
